In [ ]:
# CÉLULA 1: carrega as músicas e aplica os filtros

import json, random, re
import pretty_midi, soundfile
from IPython.display import Audio, display

dados = json.load(open("tunes.json", "r", encoding="utf-8"))
ref_notas = {'C': 60, 'D': 62, 'E': 64, 'F': 66, 'G': 67, 'A': 69, 'B': 71}

tipo_escolhido = "polka"; meter_escolhido = "2/4"; modo_escolhido = "Dminor"
# tipo_escolhido = "hornpipe"; meter_escolhido = "4/4"; modo_escolhido = "Dmixolydian"
# tipo_escolhido = "waltz"; meter_escolhido = "3/4"; modo_escolhido = "Eminor"

# tipo: "jig", "reel", "hornpipe", "polka", "strathspey", "waltz", "slide", "slip jig"
# meter: "2/4" (polka), "3/4" (waltzes) , "4/4" (reels, hornpipes, strathspeys), "6/8" (jigs, slides), "9/8" (slip jigs)
# modo: "Gmajor", "Dmajor", "Amajor", "Cmajor", "Aminor", "Eminor", "Dminor", "Edorian", "Adorian", "Ddorian", "Gmixolydian", "Dmixolydian", "Amixolydian"

def extrair_notas_abc(texto):
    texto = re.sub(r'"[^"]*"', '', texto)
    notas = []

    for nota, oitava, duracao in re.findall(r'([a-zA-Z])([,\']*)([\d/]*)', texto):
        if nota.upper() not in ref_notas:
            continue

        pitch = ref_notas[nota.upper()] + (12 if nota.islower() else 0)
        pitch += 12 * (oitava.count("'") - oitava.count(","))

        dur = 0.5
        if duracao:
            if '/' in duracao:
                num = duracao.replace('/', '')
                dur /= int(num) if num else 2
            else:
                dur *= int(duracao)

        notas.append((pitch, round(dur, 2)))

    return notas

eventos = []
for t in dados:
    if t["type"] == tipo_escolhido and t["meter"] == meter_escolhido and t.get("mode") == modo_escolhido:
        notas = extrair_notas_abc(t["abc"])
        if notas:
            eventos.append(notas)

print("Músicas extraídas:", len(eventos))

In [ ]:
# CÉLULA 2: define a ordem e monta as transições

ordem = 2

def markov(corpus, ordem):
    transicoes = {}
    iniciais = []

    for sequencia in corpus:
        if len(sequencia) <= ordem:
            continue

        iniciais.append(tuple(sequencia[:ordem]))

        for i in range(len(sequencia) - ordem):
            estado = tuple(sequencia[i:i + ordem])
            proximo = sequencia[i + ordem]

            if estado not in transicoes:
                transicoes[estado] = {}

            transicoes[estado][proximo] = transicoes[estado].get(proximo, 0) + 1

    for estado in transicoes:
        total = sum(transicoes[estado].values())
        for proximo in transicoes[estado]:
            transicoes[estado][proximo] /= total

    return transicoes, iniciais

transicoes, iniciais = markov(eventos, ordem)

print("Ordem:", ordem)
print("Estados:", len(transicoes))

In [ ]:
# CÉLULA 3: escolha quantas músicas serão geradas

quantidade_musicas = 3

def escolher_proximo(opcoes):
    sorteio = random.random()
    acumulado = 0

    for evento, probabilidade in opcoes.items():
        acumulado += probabilidade
        if sorteio <= acumulado:
            return evento

def gerar_musica(transicoes, iniciais, ordem):
    estado = list(random.choice(iniciais))
    musica = estado.copy()

    while len(musica) < 96:
        chave = tuple(estado)
        if chave not in transicoes:
            estado = list(random.choice(iniciais))
            continue

        proximo = escolher_proximo(transicoes[chave])
        musica.append(proximo)
        estado = estado[1:] + [proximo]

    return musica

musicas = []
for _ in range(quantidade_musicas):
    musicas.append(gerar_musica(transicoes, iniciais, ordem))

print("Músicas geradas:", len(musicas))

In [ ]:
# CÉLULA 4: salva as melodias em arquivos MIDI

def exportar_midi(sequencia, arquivo, bpm=115):
    midi = pretty_midi.PrettyMIDI(initial_tempo=bpm)
    instrumento = pretty_midi.Instrument(program=0)
    tempo = 0

    for pitch, duracao in sequencia:
        inicio = tempo
        fim = inicio + duracao * 60 / bpm
        instrumento.notes.append(
            pretty_midi.Note(velocity=90, pitch=pitch, start=inicio, end=fim)
        )
        tempo = fim

    midi.instruments.append(instrumento)
    midi.write(arquivo)

for i, musica in enumerate(musicas, 1):
    exportar_midi(musica, f"musica_{i}.mid")

In [ ]:
# CÉLULA 5: converte os arquivos em áudio

for i in range(1, quantidade_musicas + 1):
    midi = pretty_midi.PrettyMIDI(f"musica_{i}.mid")
    audio = midi.synthesize()
    soundfile.write(f"musica_{i}.wav", audio, 44100)

    print(f"Música {i}")
    display(Audio(audio, rate=44100))